In [1]:
import numpy as np
import pandas as pd 
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split 
from sklearn.preprocessing import StandardScaler 
from sklearn.preprocessing import LabelEncoder 

In [2]:
df = pd.read_csv("breast-cancer.csv")
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [3]:
df.drop(columns = ['id'], inplace = True)
df.head()

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [4]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size = 0.2)


In [5]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

In [6]:
encoder = LabelEncoder()
y_train_encoder = encoder.fit_transform(y_train)
y_test_encoder = encoder.transform(y_test)

In [7]:
X_train_tensor = torch.from_numpy(X_train).float()
X_test_tensor = torch.from_numpy(X_test).float()
y_train_tensor = torch.from_numpy(y_train_encoder).float()
y_test_tensor = torch.from_numpy(y_test_encoder).float()


CustomDataset Class
=

In [8]:
from torch.utils.data import Dataset, DataLoader 

class CustomDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

Dataset Object
=

In [9]:
train_dataset = CustomDataset(X_train_tensor, y_train_tensor)
test_dataset = CustomDataset(X_test_tensor, y_test_tensor)


DataLoader Object
=

In [10]:
train_loader = DataLoader(train_dataset, batch_size= 32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size= 32, shuffle=True)

Define Model
=

In [11]:
class MySimpleNN(nn.Module):
    def __init__(self, num_features):
        super(). __init__()
        self.network = nn.Sequential(
            nn.Linear(num_features, 1),
            nn.Sigmoid()
        )

    def forward(self, features):
        out = self.network(features)
        return out

Important Parameters
=

In [12]:
learning_rate = 0.1
epochs = 25

Model Creation
=

In [13]:
# Create model 
model = MySimpleNN(X_train_tensor.shape[1])

# Define optimizer
optimizer = torch.optim.SGD(model.parameters(), lr = learning_rate)

# Loss function
loss_function = nn.BCELoss()

Training Pipeline
=

In [14]:
for epochs in range(epochs):
    for batch_features, batch_labels, in train_loader:
            
          # Forward pass
            y_pred = model(X_train_tensor)
        
            # Loss calculation
            loss = loss_function(y_pred, y_train_tensor.view(-1, 1))
        
            # Clear Gradient
            optimizer.zero_grad()
        
            # Backward pass
            loss.backward()
        
            # Update parameters
            optimizer.step()
        
            print(f"Epochs : {epochs + 1}, Loss : {loss.item()}")
        
  

Epochs : 1, Loss : 0.672440230846405
Epochs : 1, Loss : 0.5194019079208374
Epochs : 1, Loss : 0.4361268877983093
Epochs : 1, Loss : 0.3836078345775604
Epochs : 1, Loss : 0.3467758595943451
Epochs : 1, Loss : 0.31917524337768555
Epochs : 1, Loss : 0.2975385785102844
Epochs : 1, Loss : 0.2800111174583435
Epochs : 1, Loss : 0.265453040599823
Epochs : 1, Loss : 0.25312089920043945
Epochs : 1, Loss : 0.24250730872154236
Epochs : 1, Loss : 0.23325280845165253
Epochs : 1, Loss : 0.2250949740409851
Epochs : 1, Loss : 0.2178371101617813
Epochs : 1, Loss : 0.21132856607437134
Epochs : 2, Loss : 0.2054518163204193
Epochs : 2, Loss : 0.20011353492736816
Epochs : 2, Loss : 0.1952386349439621
Epochs : 2, Loss : 0.19076581299304962
Epochs : 2, Loss : 0.1866445243358612
Epochs : 2, Loss : 0.18283258378505707
Epochs : 2, Loss : 0.17929457128047943
Epochs : 2, Loss : 0.17600038647651672
Epochs : 2, Loss : 0.17292429506778717
Epochs : 2, Loss : 0.17004422843456268
Epochs : 2, Loss : 0.16734099388122559
E

Evaluation
=

In [15]:
# Model evaluation using test_loader
model.eval() # set the model to evaluation mode
accuracy_list = []

with torch.no_grad():
    for batch_features, batch_labels in test_loader:

        # Forwad pass 
        y_pred = model(batch_features)
        y_pred = (y_pred > 0.5).float()


        # calculate accuracy for the current batch
        batch_accuracy = (y_pred.view(-1) == batch_labels).float().mean().item()
        accuracy_list.append(batch_accuracy)


# Calculate overall accuracy 
overall_accuracy = sum(accuracy_list) / len(accuracy_list)
print(f"Accuracy : {overall_accuracy : .4f}")

Accuracy :  0.9922
